In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving clean_data_after_eda.xlsx to clean_data_after_eda.xlsx


In [4]:
# Google Colab / Jupyter
df = pd.read_excel('clean_data_after_eda.xlsx')

# Make sure date columns are datetime
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("Shape:", df.shape)
df.head()

Shape: (14606, 44)


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,var_year_price_off_peak_var,var_year_price_peak_var,var_year_price_mid_peak_var,var_year_price_off_peak_fix,var_year_price_peak_fix,var_year_price_mid_peak_fix,var_year_price_off_peak,var_year_price_peak,var_year_price_mid_peak,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,0,1.78,0.114481,0.098142,40.606701,t,0.00,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,0.000061,2.627605e-05,0.000440,1.102785,49.550703,22.022535,1.102846,4.955073e+01,22.022975,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,0,16.27,0.145711,0.000000,44.311378,f,0.00,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0.000005,6.089453e-04,0.000000,0.006465,0.000000,0.000000,0.006470,6.089453e-04,0.000000,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,0,0,38.72,0.165794,0.087899,44.311378,f,0.00,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0.000006,2.558511e-07,0.000000,0.007662,0.000000,0.000000,0.007668,2.558511e-07,0.000000,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,0,0,19.83,0.146694,0.000000,44.311378,f,0.00,30.22,30.22,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0.000005,0.000000e+00,0.000000,0.006465,0.000000,0.000000,0.006470,0.000000e+00,0.000000,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,526,0,131.73,0.116900,0.100015,40.606701,f,52.32,44.91,44.91,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0.000015,3.552481e-06,0.000003,0.005429,0.001954,0.000869,0.005444,1.957971e-03,0.000871,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


In [5]:
# Keep the customer ID separately for reference
customer_id = df['id'].copy()

# Drop identifier / potential leakage column
df = df.drop(columns=['id', 'date_end'], errors='ignore')

print("New shape:", df.shape)

New shape: (14606, 42)


In [6]:
# Calendar features
df['activation_year'] = df['date_activ'].dt.year
df['activation_month'] = df['date_activ'].dt.month
df['activation_quarter'] = df['date_activ'].dt.quarter

# Contract / customer lifecycle features
df['days_since_product_change'] = (
    df['date_renewal'] - df['date_modif_prod']
).dt.days

df['days_to_renewal'] = (
    df['date_renewal'] - df['date_activ']
).dt.days

# Convert to approximate months
df['months_to_renewal'] = df['days_to_renewal'] / 30

# Date columns are now represented by engineered features
df = df.drop(columns=['date_activ', 'date_modif_prod'], errors='ignore')

In [7]:
# Avoid division by zero
eps = 1e-6

df['consumption_forecast_gap'] = (
    df['cons_12m'] - df['forecast_cons_12m']
)

df['consumption_forecast_ratio'] = (
    df['cons_12m'] / (df['forecast_cons_12m'] + eps)
)

df['recent_consumption_ratio'] = (
    df['cons_last_month'] /
    ((df['cons_12m'] / 12) + eps)
)

df['forecast_year_gap'] = (
    df['forecast_cons_year'] - df['forecast_cons_12m']
)

In [8]:
# Customer value / relationship features
df['margin_per_product'] = (
    df['net_margin'] / (df['nb_prod_act'] + eps)
)

df['gross_to_net_margin_ratio'] = (
    df['margin_gross_pow_ele'] /
    (df['margin_net_pow_ele'].abs() + eps)
)

df['consumption_per_product'] = (
    df['cons_12m'] / (df['nb_prod_act'] + eps)
)

# Customer with multiple products
df['multi_product_customer'] = (
    df['nb_prod_act'] > 1
).astype(int)

In [9]:
# Average annual price variation
annual_price_cols = [
    'var_year_price_off_peak_var',
    'var_year_price_peak_var',
    'var_year_price_mid_peak_var',
    'var_year_price_off_peak_fix',
    'var_year_price_peak_fix',
    'var_year_price_mid_peak_fix',
    'var_year_price_off_peak',
    'var_year_price_peak',
    'var_year_price_mid_peak'
]

six_month_price_cols = [
    'var_6m_price_off_peak_var',
    'var_6m_price_peak_var',
    'var_6m_price_mid_peak_var',
    'var_6m_price_off_peak_fix',
    'var_6m_price_peak_fix',
    'var_6m_price_mid_peak_fix',
    'var_6m_price_off_peak',
    'var_6m_price_peak',
    'var_6m_price_mid_peak'
]

df['avg_annual_price_change'] = df[annual_price_cols].mean(axis=1)
df['avg_6m_price_change'] = df[six_month_price_cols].mean(axis=1)

# Magnitude of price movement, regardless of direction
df['abs_annual_price_change'] = df[annual_price_cols].abs().mean(axis=1)
df['abs_6m_price_change'] = df[six_month_price_cols].abs().mean(axis=1)

# Simple price pressure indicator
df['price_change_acceleration'] = (
    df['avg_6m_price_change'] - df['avg_annual_price_change']
)

In [10]:
df['avg_forecast_energy_price'] = (
    df['forecast_price_energy_off_peak'] +
    df['forecast_price_energy_peak']
) / 2

df['total_forecast_price_index'] = (
    df['avg_forecast_energy_price'] +
    df['forecast_price_pow_off_peak']
)

df['has_discount'] = (
    df['forecast_discount_energy'] > 0
).astype(int)

In [11]:
new_features = [
    'activation_year', 'activation_month', 'activation_quarter',
    'days_since_product_change', 'days_to_renewal', 'months_to_renewal',
    'consumption_forecast_gap', 'consumption_forecast_ratio',
    'recent_consumption_ratio', 'forecast_year_gap',
    'margin_per_product', 'gross_to_net_margin_ratio',
    'consumption_per_product', 'multi_product_customer',
    'avg_annual_price_change', 'avg_6m_price_change',
    'abs_annual_price_change', 'abs_6m_price_change',
    'price_change_acceleration',
    'avg_forecast_energy_price', 'total_forecast_price_index',
    'has_discount'
]

display(df[new_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
activation_year,14606.0,2.010578e+03,1.653268e+00,2003.000000,2.010000e+03,2011.000000,2012.000000,2.014000e+03
activation_month,14606.0,6.558880e+00,3.514151e+00,1.000000,3.000000e+00,7.000000,10.000000,1.200000e+01
activation_quarter,14606.0,2.515952e+00,1.136425e+00,1.000000,1.000000e+00,3.000000,4.000000,4.000000e+00
days_since_product_change,14606.0,9.297707e+02,9.232672e+02,-878.000000,5.300000e+01,731.000000,1827.000000,4.395000e+03
days_to_renewal,14606.0,1.634962e+03,6.085653e+02,364.000000,1.100000e+03,1469.000000,1925.500000,4.430000e+03
months_to_renewal,14606.0,5.449872e+01,2.028551e+01,12.133333,3.666667e+01,48.966667,64.183333,1.476667e+02
consumption_forecast_gap,14606.0,1.573517e+05,5.730070e+05,-3534.240000,4.961927e+03,12605.765000,37462.037500,6.207019e+06
consumption_forecast_ratio,14606.0,2.180167e+09,5.194751e+10,0.000000,6.773176e+00,9.745389,21.927754,3.550287e+12
recent_consumption_ratio,14606.0,9.190880e-01,1.026379e+00,0.000000,0.000000e+00,0.865063,1.346755,1.531579e+01
forecast_year_gap,14606.0,-4.688520e+02,2.490632e+03,-26164.870000,-9.623325e+02,-326.915000,10.237500,1.561892e+05


In [12]:
feature_corr = (
    df[new_features + ['churn']]
    .corr(numeric_only=True)['churn']
    .drop('churn')
    .sort_values(key=abs, ascending=False)
)

display(feature_corr.to_frame('correlation_with_churn').head(15))

,correlation_with_churn
activation_year,0.075583
months_to_renewal,-0.073634
days_to_renewal,-0.073634
days_since_product_change,-0.052821
consumption_forecast_gap,-0.046059
consumption_per_product,-0.040294
margin_per_product,0.034518
gross_to_net_margin_ratio,-0.030509
avg_forecast_energy_price,0.024882
avg_annual_price_change,0.018099


In [13]:
key_features = [
    'avg_annual_price_change',
    'avg_6m_price_change',
    'abs_6m_price_change',
    'total_forecast_price_index',
    'consumption_forecast_ratio',
    'recent_consumption_ratio',
    'num_years_antig',
    'nb_prod_act',
    'net_margin'
]

display(
    df.groupby('churn')[key_features]
      .mean()
      .T
      .round(3)
)

churn,0,1
avg_annual_price_change,7.890000e-01,1.129000e+00
avg_6m_price_change,6.390000e-01,9.650000e-01
abs_6m_price_change,6.390000e-01,9.650000e-01
total_forecast_price_index,4.320200e+01,4.342800e+01
consumption_forecast_ratio,2.262323e+09,1.416677e+09
recent_consumption_ratio,9.220000e-01,8.910000e-01
num_years_antig,5.037000e+00,4.634000e+00
nb_prod_act,1.296000e+00,1.260000e+00
net_margin,1.850570e+02,2.283620e+02


In [14]:
# Remove remaining raw date columns
model_df = df.drop(
    columns=['date_renewal'],
    errors='ignore'
).copy()

# Keep target at the end
cols = [c for c in model_df.columns if c != 'churn'] + ['churn']
model_df = model_df[cols]

print("Final modelling dataset:", model_df.shape)
display(model_df.head())

Final modelling dataset: (14606, 61)


,channel_sales,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,var_year_price_off_peak_var,var_year_price_peak_var,var_year_price_mid_peak_var,var_year_price_off_peak_fix,var_year_price_peak_fix,var_year_price_mid_peak_fix,var_year_price_off_peak,var_year_price_peak,var_year_price_mid_peak,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,activation_year,activation_month,activation_quarter,days_since_product_change,days_to_renewal,months_to_renewal,consumption_forecast_gap,consumption_forecast_ratio,recent_consumption_ratio,forecast_year_gap,margin_per_product,gross_to_net_margin_ratio,consumption_per_product,multi_product_customer,avg_annual_price_change,avg_6m_price_change,abs_annual_price_change,abs_6m_price_change,price_change_acceleration,avg_forecast_energy_price,total_forecast_price_index,has_discount,churn
0,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,0.00,0,0,1.78,0.114481,0.098142,40.606701,t,0.00,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,0.000061,2.627605e-05,0.000440,1.102785,49.550703,22.022535,1.102846,4.955073e+01,22.022975,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,2013,6,2,-131,738,24.600000,0.00,0.000000,0.000000,0.00,339.494830,1.0,0.000000,1,16.150345,3.241193e+01,16.150345,3.241193e+01,16.261585,0.106312,40.713012,0,1
1,MISSING,4660,0,0,189.95,0,0,16.27,0.145711,0.000000,44.311378,f,0.00,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0.000005,6.089453e-04,0.000000,0.006465,0.000000,0.000000,0.006470,6.089453e-04,0.000000,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,2009,8,3,2201,2201,73.366667,4470.05,24.532772,0.000000,-189.95,18.889981,1.0,4659.995340,0,0.001573,2.378435e-03,0.001573,2.378435e-03,0.000805,0.072855,44.384233,0,0
2,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,47.96,0,0,38.72,0.165794,0.087899,44.311378,f,0.00,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0.000006,2.558511e-07,0.000000,0.007662,0.000000,0.000000,0.007668,2.558511e-07,0.000000,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,2010,4,2,1827,1827,60.900000,496.04,11.342785,0.000000,-47.96,6.599993,1.0,543.999456,0,0.001704,8.142738e-07,0.001704,8.142738e-07,-0.001703,0.126847,44.438224,0,0
3,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,240.04,0,0,19.83,0.146694,0.000000,44.311378,f,0.00,30.22,30.22,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0.000005,0.000000e+00,0.000000,0.006465,0.000000,0.000000,0.006470,0.000000e+00,0.000000,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,2010,3,1,1827,1827,60.900000,1343.96,6.598900,0.000000,-240.04,25.459975,1.0,1583.998416,0,0.001438,7.373868e-07,0.001438,7.373868e-07,-0.001437,0.073347,44.384725,0,0
4,MISSING,4425,0,526,445.75,526,0,131.73,0.116900,0.100015,40.606701,f,52.32,44.91,44.91,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0.000015,3.552481e-06,0.000003,0.005429,0.001954,0.000869,0.005444,1.957971e-03,0.000871,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,2010,1,1,1881,1881,62.700000,3979.25,9.927089,1.426441,80.25,47.979952,1.0,4424.995575,0,0.001838,3.110570e-06,0.001838,3.110570e-06,-0.001835,0.108457,40.715159,0,0


In [15]:
# Save the final dataset for the modelling step
model_df.to_excel('feature_engineered_churn_data.xlsx', index=False)

print("Saved: feature_engineered_churn_data.xlsx")

Saved: feature_engineered_churn_data.xlsx
